# Deepfake Detection — Google Colab GPU Runner

Trains the cross-dataset deepfake detectors on a Colab GPU (T4, 16 GB).

**Before running:** Runtime ▸ Change runtime type ▸ GPU. Upload your `code` folder (with `faces/`, `ml/`,
`experiments/`) to Google Drive first — see `CLOUD_SETUP.md`.


## 1. Check GPU


In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Edit `PROJECT` to point at the folder in your Drive that holds `faces/` and `ml/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/masters_2027/code'   # <-- edit to your path
assert os.path.isdir(os.path.join(PROJECT,'ml')),    f'No ml/ in {PROJECT}'
assert os.path.isdir(os.path.join(PROJECT,'faces')), f'No faces/ in {PROJECT}'
print('Project OK:', PROJECT)

## 3. Set paths

Reading ~3 GB of faces directly from Drive is slow. We copy faces to local disk for speed; results are written
back to Drive so they survive a disconnect.


In [ ]:
import shutil, sys, subprocess
ML = os.path.join(PROJECT,'ml')
RESULTS = os.path.join(PROJECT,'results')   # saved to Drive (persistent)
os.makedirs(RESULTS, exist_ok=True)

LOCAL_FACES = '/content/faces'
if not os.path.isdir(LOCAL_FACES):
    print('Copying faces to local disk (faster training)...')
    shutil.copytree(os.path.join(PROJECT,'faces'), LOCAL_FACES)
FACES = LOCAL_FACES
print('ML     =', ML)
print('FACES  =', FACES)
print('RESULTS=', RESULTS)

## 4. Install dependencies


In [ ]:
!pip install -q 'timm>=0.9.12' 'grad-cam>=1.5.0' 'scikit-learn>=1.4.0' 'scipy>=1.12.0' seaborn
import timm, torch
print('timm', timm.__version__, '| cuda', torch.cuda.is_available())

## 5. Sanity run — one experiment

fp16 works on Colab's T4, so keep `USE_AMP=1`.


In [ ]:
env = dict(os.environ, USE_AMP='1')
cmd = [sys.executable,'run_all.py','--exp','exp1',
       '--faces',FACES,'--results',RESULTS,'--model_preset','efficientnet_b4']
subprocess.run(cmd, cwd=ML, env=env, check=True)

In [ ]:
import json
m = json.load(open(os.path.join(RESULTS,'exp1_ffpp_to_ffpp','metrics.json')))
print('video AUC :', m['video_level']['auc'], '| n_videos:', m['n_videos'])

## 6. Full pipeline (one model at a time)

Change `PRESET` and re-run for each baseline: `efficientnet_b4` → `xception` → `resnet50` → `convnext_tiny` → `vit_base`.

Colab free sessions time out (~12h / on idle). Run one model per session and let results accumulate in Drive.


In [ ]:
PRESET = 'efficientnet_b4'   # <-- change per model and re-run
env = dict(os.environ, USE_AMP='1')
cmd = [sys.executable,'run_all.py','--exp','all',
       '--faces',FACES,'--results',RESULTS,'--model_preset',PRESET]
subprocess.run(cmd, cwd=ML, env=env, check=True)
for fn in ('results_table.csv','all_results.json'):
    src=os.path.join(RESULTS,fn)
    if os.path.exists(src):
        base,ext=fn.split('.'); shutil.copy(src, os.path.join(RESULTS, f'{base}_{PRESET}.{ext}'))
print('Done:', PRESET)

## 7. Ablations (EfficientNet-B4)


In [ ]:
env = dict(os.environ, USE_AMP='1')
def run(args):
    subprocess.run([sys.executable,'run_all.py','--faces',FACES,'--results',RESULTS]+args, cwd=ML, env=env, check=True)
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_label_smoothing'])
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_progressive_unfreeze'])
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_train_augmentation'])
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_tta','--eval_only'])
for agg in ('median','max','topk'):
    run(['--exp','all','--model_preset','efficientnet_b4','--aggregation',agg,'--eval_only'])

## 8. Results

Everything is already in `RESULTS` on your Drive (checkpoints, metrics.json, failure_cases.csv, plots). Nothing else to do.


In [ ]:
import glob
for p in sorted(glob.glob(os.path.join(RESULTS,'*','metrics.json'))):
    print(p)